# Bridge: run Tiny Aya on Colab

Use a GPU runtime. First accept the terms at https://huggingface.co/CohereLabs/tiny-aya-global with your Hugging Face account. Upload a ZIP of the `lexsi-bridge` folder containing `bridge.py`, `requirements.txt`, and `data/pilot.jsonl`. Never include credentials in the ZIP. The pilot and refusal heuristic require human review; generated results do not establish model safety.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload one project ZIP'
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    for item in z.infolist():
        path = Path(item.filename)
        if path.is_absolute() or '..' in path.parts:
            raise ValueError('Unsafe ZIP path')
    z.extractall('/content/bridge-project')
paths = list(Path('/content/bridge-project').rglob('bridge.py'))
assert len(paths) == 1, 'ZIP must contain one bridge.py'
import os
os.chdir(paths[0].parent)
print('Project:', Path.cwd())

In [ ]:
%pip install -q -r requirements.txt 'transformers>=5.15' accelerate huggingface_hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!python bridge.py curate

In [ ]:
!python bridge.py run-local --condition baseline --out results/baseline.json

In [ ]:
!python bridge.py run-local --condition guided --out results/guided.json

In [ ]:
!python bridge.py report results/baseline.json results/guided.json

Review every response for safety and usefulness independently. Record the model revision, runtime and reviewer decisions. Download results and curate a publishable, reviewed dataset before claiming a measured outcome.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/bridge-results', 'zip', 'results')
files.download('/content/bridge-results.zip')